In [3]:
import numpy as np
import sys
import scipy

from scipy.integrate import solve_ivp

import matplotlib.pyplot as plt
from matplotlib import cm

# Neutron Diffusion in a Bare Spherical Fissile Assembly

**M6023 Advanced Quantitative Modelling: Assessment 3**

**Student ID:** 23360932014

This notebook develops a one-group neutron diffusion model for a bare spherical fissile assembly.

## Contents

1. Background


---

## 1. Background

### Why does critical mass exist?
In July 1945, in the New Mexico desert, a sphere of plutonium that was roughly the size of a grapefruit was utterly compressed by a precisely timed arrangement of conventional explosives. Within microseconds, the compression pushed the plutonium past a particular threshold: the neutrons inside it stopped leaking away faster than they were being produced, and instead the population of neutrons began to grow. Each fission event released neutrons that struck other nuclei and caused further fissions, which released even more neutrons, and so on. The resulting chain reaction released the energy of roughly twenty thousand tonnes of TNT.

We have a name for what was just described: **criticality**. And the quantity that determined whether the sphere was above or below the threshold of criticality is one of the most consequential numbers in twentieth-century physics: known as the **critical mass**.

Below this threshold, a lump of fissile material is simply inert. A few neutrons may wander here and there through it, causing a few fissions, but most of the neutrons subsequently created escape out of the surface before they can find another nucleus to split. Above the threshold, the same material sustains a chain reaction. The difference between "inert lump" and "exponentially growing neutron population" is actually a matter of geometry: a sphere of radius $R$ just slightly larger than some critical value $R_c$ is qualitatively different from one just slightly smaller, even though the material itself is identical.

So, the question this notebook will attempt to answer is: **given a fissile material with known microscopic properties, what is $R_c$?**

### What is the physical picture?

Right, so why does this threshold exist at all? To understand that, we need to think about what actually happens to a neutron inside a lump of fissile material. It can meet one of three fates:

1. It gets **absorbed** by a nucleus without causing a fission at all. This is like a dud interaction, as far as the chain reaction is concerned. The neutron is just gone to the "wind".

2. It causes a **fission**, in which case the nucleus splits and releases on average $\nu$ new neutrons (typically between 2.4 and 2.9, depending on our isotope). This is the productive outcome that is typically searched for in industrial reactions. 

3. It travels all the way to the **surface** of the material and escapes into the surrounding vacuum, lost forever and forgotten.

Whether the neutron population grows, holds steady, or dies out really depends entirely on the balance between these three fates. Fission, as a process, creates neutrons at a rate proportional to how many are already present and how likely each one is to cause a fission. Absorption removes them at a rate proportional to how many are present and how likely each is to be absorbed. Pretty straightforward, right?

Maybe not. You see, leakage (neutrons escaping) works differently. It depends not on the total number of neutrons but actually on how they are distributed through the material. Neutrons only leak from regions near the surface, because a neutron deep in the interior is unlikely to make it to the outside without first bumping into something (probably another neutron). 

So, what does this mean? Well, it means leakage scales roughly with **surface area**, while production and absorption scale with **volume**.

As such, when a sphere gets larger, volume grows as $R^3$ but surface area only grows as $R^2$. So the *fraction* of neutrons that can escape decreases as the sphere gets bigger. Below a certain size, leakage just dominates and the population of neutrons dies out. Above it, production dominates and the population grows. At exactly $R_c$, the two balance. 

This is why criticality depends on geometry as much as on material itself.

### Why is this a PDE problem?

So, we've established that criticality depends on a balance in the third demnsion: neutrons near the surface leak out, neutrons in the interior cause fissions. 

But how do we actually turn that into mathematics? And why do we care? 

Well, we need a quantity that captures how neutrons are distributed across space and time. This is the **neutron flux** $\varphi(\mathbf{r}, t)$, which has units of neutrons per square centimetre per second and measures. Or, in looser terms, how many neutrons are passing through a given point per unit time. By the way, this is a field, meaning it will take a value at every point in space and every instant in time.

The three physical processes that we talked about earlier correspond to three mathematical terms governing how $\varphi$ evolves:

- **Production and absorption** act locally, at each point in our field, and depend on the flux (neutron "traffic") at that point.
- **Leakage** acts through gradients in space: neutrons tend to flow from regions of high flux to regions of low flux, like how heat flows from hot regions to cold. This is our **diffusion** mechanism, and we describe it mathematically as the Laplacian operator $\nabla^2$.

This Laplacian is a spatial second derivative. What it does is that it measures how much a quantity at a point differs from the average of its neighbours. If you've done the heat equatione literally ever (and god knows I have), you'll know this is the exact same mechanism. The only tangible difference here is that on top of diffusion, we also have a reactive source term: the material is producing and absorbing neutrons as well as letting them diffuse. The fission process!

So, when we put these three terms together, we get a PDE of the form:

$$\frac{\partial \varphi}{\partial t} = \underbrace{D \nabla^2 \varphi}_{\text{leakage (diffusion)}} + \underbrace{(\nu \Sigma_f - \Sigma_a)\varphi}_{\text{net production}}$$

wherein $D$ is the diffusion coefficient, $\Sigma_f$ and $\Sigma_a$ are the fission and absorption cross-sections (on a macroscopic level at least), and $\nu$ is the average number of neutrons released per fission.

THIS is why we need a PDE and not just an ODE: our leakage term involves *spatial* derivatives of $\varphi$, while that time evolution involves a *time* derivative. How these two interact and go back-and-forth is what produces criticality under the umbrella of geometrics. 

If we derive this equation properly, and then solve it for a spherical geometry, we can use it to calculate $R_c$. 

Which is the subject of this notebook.

### A bare sphere? One-group theory? What are we talking about?

Alright, before we begin, we  honestly need to do a quick explanation for two major simplifications we're about to make.

The **bare sphere** (a sphere of fissile material surrounded by nothing but vacuum) is, quite literally, the simplest geometry we can use to capture the essential physics. It has a single length scale (the radius), actually has an exact formula that we can solve directly (thus we can use it to check if our numerical answers are right), and it gives us an upper bound on the critical radius for a given material. Being honest, any real-world assembly with a reflector of uranium or beryllium surrounding the fissile core would have a much *smaller* critical radius than the bare case. Also, fun fact, the bare-sphere calculation we're making also is historically rich: the earliest fission device designs basically did this calculation (well, theirs was a touch more complicated, but, who's counting)

**The one-group diffusion theory** treats all neutrons as having the same effective energy. The reason we do this is basically because otherwise, we would have to do a SEVEN-dimensional problem (three spatial, one temporal, one energy, two angular - no thank you) to just two dimensions ($r$ and $t$) under spherical symmetry. You could use multi-group diffusion or even, god forbid, the full neutron transport equation, but those wouldn't really change things for our bare sphere. So, it would just be making things more complicated, and one-group theory gives us values with 10 or 20 per cent of the experimental values. And even better, the sources of that error are well documented. 

We will discuss both of these choices in Section 2, Assumptions, and a likely section on Limitations. 